## ANALYSIS OF CHICAGO TRAFFIC CRASHES

### This is a project to predict the primary factor that majorly contributes to most car accidents in Chicago. 

In [1]:
#import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

In [2]:
#load dataset and check columns
df_crashes = pd.read_csv('Traffic_Crashes_-_Crashes_20250710.csv')
df_people = pd.read_csv('Traffic_Crashes_-_People_20250710.csv')
df_vehicles = pd.read_csv('Traffic_Crashes_-_Vehicles_20250710.csv')


In [3]:
#maintain the necessary columns then merge the datasets
df_crashes = df_crashes[["CRASH_RECORD_ID", "POSTED_SPEED_LIMIT", "TRAFFIC_CONTROL_DEVICE", "DEVICE_CONDITION", "WEATHER_CONDITION", "LIGHTING_CONDITION", "CRASH_TYPE", "TRAFFICWAY_TYPE", "DAMAGE", "ALIGNMENT", "ROADWAY_SURFACE_COND", "NUM_UNITS", "MOST_SEVERE_INJURY", "INJURIES_TOTAL", "INJURIES_FATAL", "INJURIES_INCAPACITATING", "INJURIES_NON_INCAPACITATING", "INJURIES_REPORTED_NOT_EVIDENT", "INJURIES_NO_INDICATION", "INJURIES_UNKNOWN", "CRASH_HOUR", "CRASH_DAY_OF_WEEK", "CRASH_MONTH", 'PRIM_CONTRIBUTORY_CAUSE']]
df_people = df_people[["PERSON_ID", "PERSON_TYPE", "CRASH_RECORD_ID", "VEHICLE_ID", "CITY", "STATE", "SEX", "AGE", "SAFETY_EQUIPMENT", "AIRBAG_DEPLOYED","INJURY_CLASSIFICATION","PHYSICAL_CONDITION"]]
df_vehicles = df_vehicles[["CRASH_UNIT_ID", "CRASH_RECORD_ID", "UNIT_NO", "UNIT_TYPE", "MAKE", "MODEL", "VEHICLE_YEAR", "VEHICLE_DEFECT", "VEHICLE_TYPE", "VEHICLE_USE", "OCCUPANT_CNT"]]

# Merge the datasets
df_merged = pd.merge(df_crashes, df_people, on="CRASH_RECORD_ID", how="left")
df = pd.merge(df_merged, df_vehicles, on="CRASH_RECORD_ID", how="left")

In [4]:
df.head()

,CRASH_RECORD_ID,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,CRASH_TYPE,TRAFFICWAY_TYPE,DAMAGE,ALIGNMENT,...,CRASH_UNIT_ID,UNIT_NO,UNIT_TYPE,MAKE,MODEL,VEHICLE_YEAR,VEHICLE_DEFECT,VEHICLE_TYPE,VEHICLE_USE,OCCUPANT_CNT
0,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986728,1,DRIVER,JEEP,COMPASS,2023.0,NONE,PASSENGER,PERSONAL,1.0
1,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986729,2,DRIVER,AUDI,A4,2011.0,NONE,PASSENGER,PERSONAL,1.0
2,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986728,1,DRIVER,JEEP,COMPASS,2023.0,NONE,PASSENGER,PERSONAL,1.0
3,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986729,2,DRIVER,AUDI,A4,2011.0,NONE,PASSENGER,PERSONAL,1.0
4,027b0b4c21460d3441fd83929abb9673c6fc0c7d575675...,30,STOP SIGN/FLASHER,UNKNOWN,UNKNOWN,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"OVER $1,500",STRAIGHT AND LEVEL,...,2071542,1,DRIVER,UNKNOWN,OTHER (EXPLAIN IN NARRATIVE),NaN,UNKNOWN,UNKNOWN/NA,UNKNOWN/NA,1.0


In [5]:
#check the shape of the dataframe
df.shape

(4442504, 45)

In [6]:
#check for missing values 
df.isnull().sum()

CRASH_RECORD_ID                        0
POSTED_SPEED_LIMIT                     0
TRAFFIC_CONTROL_DEVICE                 0
DEVICE_CONDITION                       0
WEATHER_CONDITION                      0
LIGHTING_CONDITION                     0
CRASH_TYPE                             0
TRAFFICWAY_TYPE                        0
DAMAGE                                 0
ALIGNMENT                              0
ROADWAY_SURFACE_COND                   0
NUM_UNITS                              0
MOST_SEVERE_INJURY                  3523
INJURIES_TOTAL                      3496
INJURIES_FATAL                      3496
INJURIES_INCAPACITATING             3496
INJURIES_NON_INCAPACITATING         3496
INJURIES_REPORTED_NOT_EVIDENT       3496
INJURIES_NO_INDICATION              3496
INJURIES_UNKNOWN                    3496
CRASH_HOUR                             0
CRASH_DAY_OF_WEEK                      0
CRASH_MONTH                            0
PRIM_CONTRIBUTORY_CAUSE                0
PERSON_ID       

In [7]:
# since the data is large, we will drop the columns with more than 50% missing values the remaining columns will be filled with the mode or mean of the column
#drop columns with more than 50% missing values
threshold = 0.5 * len(df)
df = df.loc[:, df.isnull().sum() < threshold]

#fill the remaining missing values with the mode or mean of the column
for column in df.columns:
    if df[column].dtype == 'object': # categorical data
        df[column].fillna(df[column].mode()[0], inplace=True)
    else: # numerical data
        df[column].fillna(df[column].mean(), inplace=True)

In [8]:
df.isnull().sum()  # check if there are any missing values left

CRASH_RECORD_ID                  0
POSTED_SPEED_LIMIT               0
TRAFFIC_CONTROL_DEVICE           0
DEVICE_CONDITION                 0
WEATHER_CONDITION                0
LIGHTING_CONDITION               0
CRASH_TYPE                       0
TRAFFICWAY_TYPE                  0
DAMAGE                           0
ALIGNMENT                        0
ROADWAY_SURFACE_COND             0
NUM_UNITS                        0
MOST_SEVERE_INJURY               0
INJURIES_TOTAL                   0
INJURIES_FATAL                   0
INJURIES_INCAPACITATING          0
INJURIES_NON_INCAPACITATING      0
INJURIES_REPORTED_NOT_EVIDENT    0
INJURIES_NO_INDICATION           0
INJURIES_UNKNOWN                 0
CRASH_HOUR                       0
CRASH_DAY_OF_WEEK                0
CRASH_MONTH                      0
PRIM_CONTRIBUTORY_CAUSE          0
PERSON_ID                        0
PERSON_TYPE                      0
VEHICLE_ID                       0
CITY                             0
STATE               

In [9]:
df.shape

(4442504, 45)

In [10]:
df.head()

,CRASH_RECORD_ID,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,CRASH_TYPE,TRAFFICWAY_TYPE,DAMAGE,ALIGNMENT,...,CRASH_UNIT_ID,UNIT_NO,UNIT_TYPE,MAKE,MODEL,VEHICLE_YEAR,VEHICLE_DEFECT,VEHICLE_TYPE,VEHICLE_USE,OCCUPANT_CNT
0,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986728,1,DRIVER,JEEP,COMPASS,2023.000000,NONE,PASSENGER,PERSONAL,1.0
1,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986729,2,DRIVER,AUDI,A4,2011.000000,NONE,PASSENGER,PERSONAL,1.0
2,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986728,1,DRIVER,JEEP,COMPASS,2023.000000,NONE,PASSENGER,PERSONAL,1.0
3,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"$501 - $1,500",STRAIGHT AND LEVEL,...,1986729,2,DRIVER,AUDI,A4,2011.000000,NONE,PASSENGER,PERSONAL,1.0
4,027b0b4c21460d3441fd83929abb9673c6fc0c7d575675...,30,STOP SIGN/FLASHER,UNKNOWN,UNKNOWN,DAYLIGHT,NO INJURY / DRIVE AWAY,DIVIDED - W/MEDIAN (NOT RAISED),"OVER $1,500",STRAIGHT AND LEVEL,...,2071542,1,DRIVER,UNKNOWN,OTHER (EXPLAIN IN NARRATIVE),2014.012677,UNKNOWN,UNKNOWN/NA,UNKNOWN/NA,1.0


### Target and feature selection

In [11]:
#the dataset is too large, we will randomly sample 10000 of the data
df = df.sample(10000, random_state=42).reset_index(drop=True)


In [12]:
#scale the numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.shape

(10000, 45)

In [13]:
df.columns

Index(['CRASH_RECORD_ID', 'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE',
       'DEVICE_CONDITION', 'WEATHER_CONDITION', 'LIGHTING_CONDITION',
       'CRASH_TYPE', 'TRAFFICWAY_TYPE', 'DAMAGE', 'ALIGNMENT',
       'ROADWAY_SURFACE_COND', 'NUM_UNITS', 'MOST_SEVERE_INJURY',
       'INJURIES_TOTAL', 'INJURIES_FATAL', 'INJURIES_INCAPACITATING',
       'INJURIES_NON_INCAPACITATING', 'INJURIES_REPORTED_NOT_EVIDENT',
       'INJURIES_NO_INDICATION', 'INJURIES_UNKNOWN', 'CRASH_HOUR',
       'CRASH_DAY_OF_WEEK', 'CRASH_MONTH', 'PRIM_CONTRIBUTORY_CAUSE',
       'PERSON_ID', 'PERSON_TYPE', 'VEHICLE_ID', 'CITY', 'STATE', 'SEX', 'AGE',
       'SAFETY_EQUIPMENT', 'AIRBAG_DEPLOYED', 'INJURY_CLASSIFICATION',
       'PHYSICAL_CONDITION', 'CRASH_UNIT_ID', 'UNIT_NO', 'UNIT_TYPE', 'MAKE',
       'MODEL', 'VEHICLE_YEAR', 'VEHICLE_DEFECT', 'VEHICLE_TYPE',
       'VEHICLE_USE', 'OCCUPANT_CNT'],
      dtype='object')

In [14]:
#drop columns thats are not predictive
df.drop(columns=['PERSON_TYPE', 'INJURY_CLASSIFICATION', 'UNIT_NO', 'CRASH_TYPE', 'DAMAGE', 'NUM_UNITS', 'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH'], inplace=True)

In [15]:
#separate features and target variable
y = df['PRIM_CONTRIBUTORY_CAUSE']  # target variable
X = df.drop(columns=['PRIM_CONTRIBUTORY_CAUSE'])  # features

In [16]:
#IDENTIFY CATEGORICAL AND NUMERICAL COLUMNS
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

### Preprocessing Pipelines

In [17]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
])

In [18]:
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [19]:
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [20]:
#full pipeline with classifier
clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

### Train-test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify= y)
categorical_cols = X_train.select_dtypes(include=['object']).columns
X_train[categorical_cols] = X_train[categorical_cols].astype(str)

In [ ]:
#train the model
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer())]),
                                                  ['POSTED_SPEED_LIMIT', 'AGE',
                                                   'VEHICLE_YEAR',
                                                   'OCCUPANT_CNT']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['TRAFFIC_CONTROL_DEVICE',
                                                   'DEVICE_CONDITION',
                                                   'WEATHER_CONDITION',
                                                   'LIGHTING_CONDITION',
                                                   'TRAFFICWAY_TYPE',
                                                   'ALIGNMENT',
                                                   'ROADWAY_SURFACE_COND',
                                                   'CITY', 'STATE', 'SEX',
                                                   'SAFETY_EQUIPMENT',
                                                   'AIRBAG_DEPLOYED',
                                                   'PHYSICAL_CONDITION',
                                                   'UNIT_TYPE', 'MAKE', 'MODEL',
                                                   'VEHICLE_DEFECT',
                                                   'VEHICLE_TYPE',
                                                   'VEHICLE_USE'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [ ]:
#predict and evaluate the model
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

                                                                                  precision    recall  f1-score   support

                                                                          ANIMAL       0.00      0.00      0.00         1
                                          BICYCLE ADVANCING LEGALLY ON RED LIGHT       0.00      0.00      0.00         1
                                               CELL PHONE USE OTHER THAN TEXTING       0.00      0.00      0.00         4
                                                DISREGARDING OTHER TRAFFIC SIGNS       0.00      0.00      0.00         7
                                                      DISREGARDING ROAD MARKINGS       0.00      0.00      0.00         3
                                                          DISREGARDING STOP SIGN       0.33      0.03      0.06        29
                                                    DISREGARDING TRAFFIC SIGNALS       0.28      0.09      0.14        54
                       

In [ ]:
#From above classifier, Random Forest the accuracy  is too low,
#drop rows with missing target
df = df.dropna(subset=['PRIM_CONTRIBUTORY_CAUSE'])

In [ ]:
#focus on top 10 contributory causes
top_cause = df['PRIM_CONTRIBUTORY_CAUSE'].value_counts().nlargest(10).index
df = df[df['PRIM_CONTRIBUTORY_CAUSE'].isin(top_cause)]

In [ ]:
#separate features and target
y = df['PRIM_CONTRIBUTORY_CAUSE']
X = df.drop(columns=['PRIM_CONTRIBUTORY_CAUSE'])

In [ ]:
#Identify categorical and numerical cols
categorical_cols=X.select_dtypes(include= ['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:

clf2 = Pipeline([
    ('preprocessor', preprocessor),
    ('Classifier', XGBClassifier(use_label_encoder = False, eval_metric = 'mlogloss'))
])

In [ ]:
#processing pipelines
numeric_transformer = Pipeline([
('imputer', SimpleImputer(strategy= 'median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1))
])


preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [ ]:
clf2 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(use_label_encoder = False, eval_metric = 'mlogloss'))
])

In [ ]:
#train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

In [ ]:
#fit the model
clf2.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['POSTED_SPEED_LIMIT', 'AGE',
                                                   'VEHICLE_YEAR',
                                                   'OCCUPANT_CNT']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))])...
                               interaction_constraints='',
                               learning_rate=0.300000012, max_delta_step=0,
                               max_depth=6, min_child_weight=1, missing=nan,
                               monotone_constraints='()', n_estimators=100,
                               n_jobs=0, num_parallel_tree=1,
                               objective='multi:softprob', random_state=0,
                               reg_alpha=0, reg_lambda=1, scale_pos_weight=None,
                               subsample=1, tree_method='exact',
                               use_label_encoder=False, validate_parameters=1,
                               verbosity=None))])

In [ ]:
#predict and evaluate the model
y_pred = clf2.predict(X_test)
print(classification_report(y_pred, y_test))

                                        precision    recall  f1-score   support

   DRIVING SKILLS/KNOWLEDGE/EXPERIENCE       0.00      0.00      0.00         6
FAILING TO REDUCE SPEED TO AVOID CRASH       0.00      0.00      0.00        23
         FAILING TO YIELD RIGHT-OF-WAY       0.23      0.31      0.26       194
                 FOLLOWING TOO CLOSELY       0.20      0.23      0.21       208
                      IMPROPER BACKING       0.03      0.15      0.05        13
                   IMPROPER LANE USAGE       0.01      0.14      0.03         7
           IMPROPER OVERTAKING/PASSING       0.02      0.10      0.03        20
            IMPROPER TURNING/NO SIGNAL       0.00      0.00      0.00         8
                        NOT APPLICABLE       0.01      0.05      0.02        20
                   UNABLE TO DETERMINE       0.76      0.43      0.55      1244

                              accuracy                           0.37      1743
                             macro avg